In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

### Calculating DEGs statistics

### For each LFC/FDR cutoff set we get diferent set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

### Biotypes

https://www.ensembl.org/info/genome/genebuild/biotypes.html

### Ensembl Gallery

https://www.ensembl.org/info/website/gallery.html

In [ ]:
import os, sys, pickle, yaml
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

from Basic import *
from MTD_lib import *
# from config_lib import *
# from stat_lib import *

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

In [ ]:
root_chibe = dic_yml['root_chibe']
root_colab = dic_yml['root_colab']
root0 = dic_yml['root0']

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

disease = dic_yml['disease']
gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

abs_lfc_cutoff_inf = dic_yml['abs_lfc_cutoff_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
num_min_degs_for_ptw_enr = dic_yml['num_min_degs_for_ptw_enr']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_index = dic_yml['saturation_lfc_index']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(project, s_project, case_list, root0)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1
abs_lfc_cutoff, fdr_lfc_cutoff, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={abs_lfc_cutoff:.3f}; fdr={fdr_lfc_cutoff:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

In [ ]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, num_min_degs_for_ptw_enr=num_min_degs_for_ptw_enr,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          root_colab=root_colab, root_chibe=root_chibe, 
          abs_lfc_cutoff_inf=abs_lfc_cutoff_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_index=saturation_lfc_index, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, abs_lfc_cutoff=1, fdr_lfc_cutoff=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
mtd.echo_parameters()

In [ ]:
mtd.case, mtd.group, mtd.gender, mtd.age

In [ ]:
mtd.geneset_num, mtd.geneset_lib

In [ ]:
mtd.gene.df_my_gene.head(2)

### Studying default non-conding genes

### WNT

In [ ]:
case = case_list[0]

ret, degs, degs_ensembl, dfdegs = mtd.open_case_params(case, abs_lfc_cutoff=1, fdr_lfc_cutoff=0.05, verbose=False)
print("\nEcho Parameters - default:")
mtd.echo_parameters()


In [ ]:
dflfc = mtd.dflfc
dflfc.biotype.unique()

### dflfc_noncod

In [ ]:
biotype_list = ['LNC']

dflfc_noncod = dflfc[ dflfc.biotype.isin(biotype_list)]
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)

cols = ['probe', 'symbol', 'symbol_prev', 'symb_or_syn', 'biotype', '_type', 'lfc', 'abs_lfc',
        'pval', 'fdr', 'mean_exp', 't', 'B', 'description', 'desc_gff', 'description_prev',
        'accession', 'ensembl_id', 'ensembl_transc_id', 'geneid', 'cytoband', 'symbol_pipe',
        'seqname', 'start', 'end', 'go_id', 'seq']

cols = ['probe', 'symbol', 'accession', 'geneid', 'ensembl_id', 'biotype', '_type', 'lfc', 'abs_lfc',
        'pval', 'fdr', 'description', 'desc_gff', ]

#dflfc_noncod['geneid'] = dflfc_noncod['geneid'].astype(int)
pdwritecsv(dflfc_noncod, f'LNC_table_{mtd.case}.tsv', mtd.root_nc)
dflfc_noncod[cols].head()

In [ ]:
cols = ['LNC', 'protein_coding', 'hasSymbol', 'UNK']

dflfc_noncod = dflfc[ ~dflfc.biotype.isin(cols)]
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)

dflfc_noncod.head()

### LncRNA_Disease v3.0

http://www.rnanut.net/lncrnadisease/index.php/home/

http://www.rnanut.net/lncrnadisease/

### miR2Disease

http://watson.compbio.iupui.edu:8080/miR2Disease/index.jsp

### Disease-associated Enhancer Catalog

http://biocc.hrbmu.edu.cn/DiseaseEnhancer/

In [ ]:
root_lrd = os.path.join(root_colab, 'lncRNA')
fname = 'all_ncRNA_disease_information.tsv'

fullname = os.path.join(root_lrd, fname)

if os.path.exists(fullname):
    dfdis = pdreadcsv(fname, root_lrd)
    print(len(dfdis))
else:
    dfdis = pd.DataFrame()
    
dfdis.head(3)

In [ ]:
symbols = dflfc_noncod.symbol
print("dflfc_noncod", len(symbols))
"; ".join(symbols)

### manually from genecards

TPTE2P1 (TPTE2 Pseudogene 1) is a Pseudogene. Diseases associated with TPTE2P1 include Hepatocellular Carcinoma and Colorectal Cancer.

PRSS30P (Serine Protease 30, Pseudogene) is a Pseudogene

LINC00472 (Long Intergenic Non-Protein Coding RNA 472) - Diseases associated with LINC00472 include Lung Cancer Susceptibility 3 and Squamous Cell Carcinoma, Head and Neck.

ZNF702P (Zinc Finger Protein 702, Pseudogene) - 

### maayanlab.cloud/Harmonizome/gene

https://maayanlab.cloud/Harmonizome/gene

LOC100190940 - has 673 functional associations with biological entities spanning 3 categories (molecular profile, cell line, cell type or tissue, gene, protein or microRNA) extracted from 12 datasets. https://maayanlab.cloud/Harmonizome/gene/LOC100190940


LOC440896 - has 390 functional associations with biological entities spanning 7 categories (molecular profile, organism, disease, phenotype or trait, chemical, functional term, phrase or reference, cell line, cell type or tissue, gene, protein or microRNA) extracted from 19 datasets. https://maayanlab.cloud/Harmonizome/gene/LOC440896


LOC149351 - has 344 functional associations with biological entities spanning 6 categories (organism, disease, phenotype or trait, chemical, functional term, phrase or reference, cell line, cell type or tissue, gene, protein or microRNA) extracted from 10 datasets. https://maayanlab.cloud/Harmonizome/gene/LOC149351

LOC644662 - has 252 functional associations with biological entities spanning 5 categories (molecular profile, disease, phenotype or trait, functional term, phrase or reference, cell line, cell type or tissue, gene, protein or microRNA) extracted from 9 datasets.

LOC643923 - has 1,208 functional associations with biological entities spanning 4 categories (molecular profile, disease, phenotype or trait, cell line, cell type or tissue, gene, protein or microRNA) extracted from 17 datasets.

In [ ]:
symbols_out = [x for x in symbols if x not in dfdis['ncRNA Symbol'].to_list()]
print(len(symbols_out))
"; ".join(symbols_out)

In [ ]:
dfin = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
print("dflfc_noncod symbols", len(symbols))
print("dfdis", len(dfdis))
print("df commons", len(dfin))
print("df commons unique", len(dfin['ncRNA Symbol'].unique()))
dfin

In [ ]:
symb_identified_list = dfin['ncRNA Symbol'].unique()
symb_identified_list

## Referências para estes LNC do WNT

### LINC00472

17. Seo D., Roh J., Chae Y., Kim W. Gene expression profiling after LINC00472 overexpression in an NSCLC cell line1. Cancer Biomark. Sect. A Dis. Markers. 2021;32:175–188. doi: 10.3233/CBM-210242. [PubMed] [CrossRef] [Google Scholar]

  -  Bladder carcinoma	Growth/metastasis
93. Li L, Qi F, Wang K. Matrine restrains cell growth and metastasis by up-regulating LINC00472 in bladder carcinoma. Cancer Manage Res (2020) 12:1241–51.   10.2147/cmar.S224701 [PMC free article] [PubMed] [CrossRef] [Google Scholar]

### TPTE2P1

63. Corrigendum. Downregulation of TPTE2P1 inhibits migration and invasion of gallbladder cancer cells. Chem Biol Drug Des. 2018;92:1816. [PubMed] [Google Scholar]




In [ ]:
mtd.root_nc

In [ ]:
fname = f"lnc_disease_identification_for_{mtd.case}.tsv"
pdwritecsv(dfin, fname, mtd.root_nc)

In [ ]:
print(len(dfin))
dfin[ dfin['ncRNA Symbol'] == symb_identified_list[0] ]

In [ ]:
dfin[ dfin['ncRNA Symbol'] == symb_identified_list[1] ]

In [ ]:
dflfc_LFC = dflfc[ dflfc.biotype == 'LNC']

fname = f'LNC_special_non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_LFC, fname, mtd.root_result, verbose=True)

print(len(dflfc_LFC))
dflfc_LFC.head()

In [ ]:
symbols = dflfc_LFC.symbol.unique()
len(symbols), ", ".join(symbols)

In [ ]:
df_ident_LNC = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
symb_identified_list = df_ident_LNC['ncRNA Symbol'].unique()
symb_identified_list

### G4

In [ ]:
case = case_list[1]

ret, degs, degs_ensembl, dfdegs = mtd.open_case_params(case, abs_lfc_cutoff=1, fdr_lfc_cutoff=0.05, verbose=False)
print("\nEcho Parameters - default:")
mtd.echo_parameters()

In [ ]:
dflfc = mtd.dflfc
dflfc.biotype.unique()

In [ ]:
biotype_list = ['LNC']

dflfc_noncod = dflfc[ dflfc.biotype.isin(biotype_list)]
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)

cols = ['probe', 'symbol', 'symbol_prev', 'symb_or_syn', 'biotype', '_type', 'lfc', 'abs_lfc',
        'pval', 'fdr', 'mean_exp', 't', 'B', 'description', 'desc_gff', 'description_prev',
        'accession', 'ensembl_id', 'ensembl_transc_id', 'geneid', 'cytoband', 'symbol_pipe',
        'seqname', 'start', 'end', 'go_id', 'seq']

cols = ['probe', 'symbol', 'accession', 'geneid', 'ensembl_id', 'biotype', '_type', 'lfc', 'abs_lfc',
        'pval', 'fdr', 'description', 'desc_gff', ]

#dflfc_noncod['geneid'] = dflfc_noncod['geneid'].astype(int)
pdwritecsv(dflfc_noncod, f'LNC_table_{mtd.case}.tsv', mtd.root_nc)
dflfc_noncod[cols].head()

In [ ]:
cols = ['LNC', 'protein_coding', 'hasSymbol', 'UNK']

dflfc_noncod = dflfc[ ~dflfc.biotype.isin(cols)]
fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

dflfc_noncod.head()

In [ ]:
symbols = dflfc_noncod.symbol
len(symbols), ", ".join(symbols)

In [ ]:
dfin = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
print(len(dfin))
symb_identified_list = dfin['ncRNA Symbol'].unique()
len(symb_identified_list), symb_identified_list

In [ ]:
fname = f"lnc_disease_identification_for_{mtd.case}.tsv"
pdwritecsv(dfin, fname, mtd.root_nc, verbose=True)

In [ ]:
i = 0
print(len(dfin))
dfin[ dfin['ncRNA Symbol'] == symb_identified_list[i] ]

In [ ]:
dflfc_LFC = dflfc[ dflfc.biotype == 'LNC']

fname = f'LNC_special_non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_LFC, fname, mtd.root_result, verbose=True)
print(len(dflfc_LFC))
dflfc_LFC.head()

In [ ]:
symbols = dflfc_LFC.symbol.unique()
len(symbols), ", ".join(symbols)

In [ ]:
df_ident_LNC = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
symb_identified_list = df_ident_LNC['ncRNA Symbol'].unique()
symb_identified_list